In [0]:
pip install pandas

In [0]:
import pandas

In [0]:
df = pandas.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales.csv")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame(df)


In [0]:

from pyspark.sql.functions import col, from_json, schema_of_json

# sample one JSON string
sample_json = df.select("product").filter(col("product").isNotNull()).first()[0]

# infer schema
json_schema = schema_of_json(sample_json)

# flatten
df_flat = df.withColumn("product_json", from_json(col("product"), json_schema)) \
    .select("*", "product_json.*") \
    .drop("product", "product_json")

display(df_flat)

In [0]:
df_flat.write.format("Delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("batch_1.data.sales")